# Stage 13: Dashboard artifacts and launch
The Streamlit app (`app/streamlit_app.py`) reads lightweight, pre-computed artifacts so it starts in seconds and never
needs the raw 2.5 GB data. This notebook builds those artifacts and runs a **training-serving consistency test**: rebuilding
features from raw application fields must reproduce the stored PDs exactly.

| Artifact | Used for |
|---|---|
| `models/final_model.joblib` | Scoring (pipeline + calibration + feature contract) |
| `models/decision_policy.json` | Approve / refer / decline cutoffs and risk grades |
| `data/processed/dashboard_portfolio.parquet` | Overview page: test-set PDs, outcomes, segments, decisions |
| `data/processed/dashboard_applicants.parquet` | Applicant page: raw cleaned application fields (sample) |
| `data/processed/dashboard_history.parquet` | Applicant page: historical aggregates for those applicants |
| `reports/*.csv / *.json` | Model performance, threshold sweep, SHAP importance, fairness |

In [1]:
import sys
import warnings
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from credit_risk.artifacts import load_final_model, load_split_frames
from credit_risk.config import get_paths, load_config
from credit_risk.features import assemble_features
from credit_risk.thresholds import assign_decision, assign_grade, load_policy

cfg = load_config()
paths = get_paths(cfg)
ID, TARGET = cfg["data"]["id_column"], cfg["data"]["target"]
N_APPLICANTS = 5000  # applicants available for what-if scoring in the app

train, valid, test = load_split_frames(paths)
model = load_final_model(paths)
policy = load_policy(paths["models"] / "decision_policy.json")

## 1. Portfolio table (test set)

In [2]:
pd_test = model.predict_pd(test)
portfolio = test[[ID, TARGET, "AMT_CREDIT", "AMT_INCOME_TOTAL", "AMT_ANNUITY", "NAME_CONTRACT_TYPE", "NAME_INCOME_TYPE",
                  "NAME_EDUCATION_TYPE", "AGE_GROUP", "CODE_GENDER", "OCCUPATION_TYPE", "EXT_SOURCE_MEAN"]].copy()
portfolio["PD"] = pd_test
portfolio["RISK_GRADE"] = assign_grade(pd_test, policy["risk_grades"])
portfolio["DECISION"] = assign_decision(pd_test, policy)
portfolio["EXPECTED_LOSS"] = portfolio["PD"] * policy["lgd"] * portfolio["AMT_CREDIT"]
portfolio.to_parquet(paths["processed_data"] / "dashboard_portfolio.parquet", index=False)
portfolio.groupby("DECISION").agg(applicants=(ID, "size"), default_rate=(TARGET, "mean"), mean_pd=("PD", "mean"))

,applicants,default_rate,mean_pd
DECISION,,,
Approve,38374,0.045890,0.046659
Decline,4357,0.320404,0.307170
Refer to underwriter,3396,0.166961,0.175354


## 2. Applicant sample for what-if scoring
The app lets a user edit an applicant's income, loan amount, annuity and external scores, then re-scores. To do that it
needs the *raw* cleaned application fields (to rebuild ratios) plus the *historical aggregates* (unchanged by the edit).

In [3]:
app_clean = pd.read_parquet(paths["interim_data"] / "application_train_clean.parquet")
aggregated = pd.read_parquet(paths["processed_data"] / "aggregated_features.parquet")

ids = test[ID].sample(min(N_APPLICANTS, len(test)), random_state=1)
# make sure every decision type is represented
for d in ["Decline", "Refer to underwriter", "Approve"]:
    extra = portfolio.loc[portfolio["DECISION"] == d, ID].head(50)
    ids = pd.concat([ids, extra]).drop_duplicates()
applicants = app_clean[app_clean[ID].isin(ids)].reset_index(drop=True)
history = aggregated[aggregated[ID].isin(ids)].reset_index(drop=True)
applicants.to_parquet(paths["processed_data"] / "dashboard_applicants.parquet", index=False)
history.to_parquet(paths["processed_data"] / "dashboard_history.parquet", index=False)
print(f"{len(applicants):,} applicants, {history.shape[1] - 1} historical features")

5,136 applicants, 99 historical features


## 3. Training-serving consistency test
Rebuild features from raw fields exactly as the app does and confirm the PDs match the stored ones.

In [4]:
rebuilt = assemble_features(applicants, history)
pd_rebuilt = pd.Series(model.predict_pd(rebuilt), index=rebuilt[ID])
pd_stored = portfolio.set_index(ID).loc[pd_rebuilt.index, "PD"]
max_diff = float(np.abs(pd_rebuilt - pd_stored).max())
print(f"Max |PD difference| between training pipeline and app pipeline: {max_diff:.2e}")
assert max_diff < 1e-6, "Training-serving skew detected!"
print("Consistency test passed ✓")

Max |PD difference| between training pipeline and app pipeline: 0.00e+00
Consistency test passed ✓


## 4. Launch the dashboard
From the project root, with the `credit-risk` environment active:
```bash
streamlit run app/streamlit_app.py
```
Pages: **Portfolio overview**, **Applicant scoring** (PD, risk grade, reason codes, experimental recommendation, what-if),
**Model performance**, **Fairness**.

**Disclaimer shown in the app:** this is an educational prototype trained on a public dataset. It is not a production
credit-decisioning system and must not be used to make real lending decisions.